# 자연어 처리 1일차 (2) — 서브워드 토큰화 : BPE 직접 구현

교재 `자연어처리` 교재9 p.15~26 (서브워드 · BPE 예시 hug / pug / pun / bun / hugs) + p.27 (WordPiece 학습 과정)

- 교재 표를 **코드로 한 단계씩 재현**해서 숫자가 맞는지 확인하고, 메모의 BPE 부분을 바로잡습니다.
- 실제 NSMC 로 WordPiece 사전을 만드는 것은 [`wordpiece_vocab_nsmc.ipynb`](./wordpiece_vocab_nsmc.ipynb).

**서브워드** = 단어를 더 쪼갠, 의미를 가진 최소한의 글자 조각 — 레고 블록처럼 적은 조각으로 많은 단어를 표현한다.
예) `unhappiness` → `un` + `happi` + `ness` (사전에 통째로 없어도 '행복하지 않은 상태' 를 유추)

In [1]:
# 그래프 한글 폰트 설정 (운영체제별)
import platform
import matplotlib.pyplot as plt

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

## 1. 시작 — 단어 빈도와 문자 단위 분리 (교재 p.16~17)

In [2]:
from collections import Counter

word_freq = {'hug': 10, 'pug': 5, 'pun': 12, 'bun': 4, 'hugs': 5}    # 교재 예시 (단어 : 등장 횟수)

# 모든 단어를 문자 단위로 분리 → 각 단어를 '토큰 튜플' 로 표현
splits = {w: tuple(w) for w in word_freq}
vocab = sorted({ch for w in word_freq for ch in w})                    # 초기 단어 사전 = 문자 7개
print('초기 단어 사전 :', vocab, len(vocab), '개')
for w, toks in splits.items():
    print(f'{w:5s} → {", ".join(toks):12s} 빈도 {word_freq[w]}')

초기 단어 사전 : ['b', 'g', 'h', 'n', 'p', 's', 'u'] 7 개
hug   → h, u, g      빈도 10
pug   → p, u, g      빈도 5
pun   → p, u, n      빈도 12
bun   → b, u, n      빈도 4
hugs  → h, u, g, s   빈도 5


## 2. 바이그램 쌍 세기 (교재 p.18~19)

이웃한 토큰 두 개(**바이그램**)를 묶고, **같은 쌍끼리 빈도를 합친다**. 단어 하나 안의 쌍은 그 단어의 빈도만큼 센다.

In [3]:
def pair_list(splits, word_freq):
    """교재 p.19 표 : 단어마다 바이그램 쌍과 빈도를 나열"""
    rows = []
    for w, toks in splits.items():
        for a, b in zip(toks, toks[1:]):
            rows.append(((a, b), word_freq[w], w))
    return rows


def pair_stats(splits, word_freq):
    """교재 p.20 표 : 같은 쌍끼리 빈도 합치기"""
    stats = Counter()
    for (pair, f, _) in pair_list(splits, word_freq):
        stats[pair] += f
    return stats


for pair, f, w in pair_list(splits, word_freq):
    print(f'{w:5s} {pair}  {f}')
print()
stats = pair_stats(splits, word_freq)
print('합친 빈도 :', dict(sorted(stats.items())))
best = max(stats, key=stats.get)
print('가장 많은 쌍 :', best, stats[best], '회')

hug   ('h', 'u')  10
hug   ('u', 'g')  10
pug   ('p', 'u')  5
pug   ('u', 'g')  5
pun   ('p', 'u')  12
pun   ('u', 'n')  12
bun   ('b', 'u')  4
bun   ('u', 'n')  4
hugs  ('h', 'u')  5
hugs  ('u', 'g')  5
hugs  ('g', 's')  5

합친 빈도 : {('b', 'u'): 4, ('g', 's'): 5, ('h', 'u'): 15, ('p', 'u'): 17, ('u', 'g'): 20, ('u', 'n'): 16}
가장 많은 쌍 : ('u', 'g') 20 회


교재 p.20 과 같다 — `(u, g)` 20회 = hug 10 + pug 5 + hugs 5, `(h, u)` 15회 = hug 10 + hugs 5.

> **📝 고친 부분** (메모 `hug (hu,ug) 10번 / hu 5번 / ug 5번`)
> - `hug` 하나에서 나오는 쌍은 `(h, u)` · `(u, g)` 각각 **10번**(hug 의 빈도). `hu 5번 · ug 5번` 은 hugs 에서 나온 몫이고, 합치면 `(h, u)` **15** · `(u, g)` **20**

## 3. 병합 반복 — BPE 학습 (교재 p.20~26)

1. 가장 많이 나온 쌍을 **하나로 합쳐** 단어 사전에 추가
2. 모든 단어를 새 사전으로 **다시 표시**
3. 바이그램 빈도를 다시 세서 1 로 — 원하는 사전 크기가 될 때까지

![BPE 병합 애니메이션](./images/anim_02_bpe_merges.gif)

In [4]:
def merge(splits, pair):
    """모든 단어에서 pair (a, b) 를 'ab' 하나로 합치기"""
    a, b = pair
    new = {}
    for w, toks in splits.items():
        out, i = [], 0
        while i < len(toks):
            if i < len(toks) - 1 and toks[i] == a and toks[i + 1] == b:
                out.append(a + b); i += 2
            else:
                out.append(toks[i]); i += 1
        new[w] = tuple(out)
    return new


def train_bpe(word_freq, num_merges, verbose=True):
    splits = {w: tuple(w) for w in word_freq}
    vocab = sorted({ch for w in word_freq for ch in w})
    merges = []
    for step in range(1, num_merges + 1):
        stats = pair_stats(splits, word_freq)
        if not stats:
            break
        best = max(stats, key=stats.get)                 # 빈도 최대 쌍 (동점이면 먼저 나온 쌍)
        splits = merge(splits, best)
        merges.append(best)
        vocab.append(best[0] + best[1])
        if verbose:
            print(f'[{step}] 병합 {best} ({stats[best]}회) → 사전 추가 {best[0] + best[1]!r}')
            print('    단어 표시 :', {w: ', '.join(t) for w, t in splits.items()})
            print('    단어 사전 :', vocab)
    return merges, vocab, splits


merges, vocab_bpe, final_splits = train_bpe(word_freq, num_merges=4)

[1] 병합 ('u', 'g') (20회) → 사전 추가 'ug'
    단어 표시 : {'hug': 'h, ug', 'pug': 'p, ug', 'pun': 'p, u, n', 'bun': 'b, u, n', 'hugs': 'h, ug, s'}
    단어 사전 : ['b', 'g', 'h', 'n', 'p', 's', 'u', 'ug']
[2] 병합 ('u', 'n') (16회) → 사전 추가 'un'
    단어 표시 : {'hug': 'h, ug', 'pug': 'p, ug', 'pun': 'p, un', 'bun': 'b, un', 'hugs': 'h, ug, s'}
    단어 사전 : ['b', 'g', 'h', 'n', 'p', 's', 'u', 'ug', 'un']
[3] 병합 ('h', 'ug') (15회) → 사전 추가 'hug'
    단어 표시 : {'hug': 'hug', 'pug': 'p, ug', 'pun': 'p, un', 'bun': 'b, un', 'hugs': 'hug, s'}
    단어 사전 : ['b', 'g', 'h', 'n', 'p', 's', 'u', 'ug', 'un', 'hug']
[4] 병합 ('p', 'un') (12회) → 사전 추가 'pun'
    단어 표시 : {'hug': 'hug', 'pug': 'p, ug', 'pun': 'pun', 'bun': 'b, un', 'hugs': 'hug, s'}
    단어 사전 : ['b', 'g', 'h', 'n', 'p', 's', 'u', 'ug', 'un', 'hug', 'pun']


교재와 순서 · 빈도가 모두 같다 : `ug`(20) → `un`(16) → `hug`(15) → `pun`(12).
최종 단어 사전 = `b, g, h, n, p, s, u, ug, un, hug, pun` (11개). 자주 안 나온 `pug` · `bun` · `hugs` 는 통째로는 없고 `p + ug` · `b + un` · `hug + s` 로 표현된다.

> **📝 고친 부분** (메모 바이그램 표)
> - `pun -> p,u,n -> (p,u) 12개, (u,p) 12개` → 두 번째 쌍은 **(u, n)**
> - `(h.ug)1whdfb -> 10종류` → 한/영 전환 오타(`whdfb` = 종류). **(h, ug) 10회**
> - `bun -> b,,u,n` → `b, u, n`
> - `OVV` → **OOV**

## 4. 학습한 BPE 로 새 단어 토큰화

새 단어도 **학습 때 병합한 순서대로** 똑같이 합친다. 사전에 없는 문자는 `[UNK]`.

In [5]:
def bpe_tokenize(word, merges, alphabet):
    toks = [ch if ch in alphabet else '[UNK]' for ch in word]
    for a, b in merges:                                 # 학습 때 병합 순서 그대로
        i, out = 0, []
        while i < len(toks):
            if i < len(toks) - 1 and toks[i] == a and toks[i + 1] == b:
                out.append(a + b); i += 2
            else:
                out.append(toks[i]); i += 1
        toks = out
    return toks


alphabet = set('bghnpsu')
for w in ['hug', 'hugs', 'pug', 'bun', 'bug', 'pugs', 'unhug', 'mug', 'thug']:
    print(f'{w:6s} → {bpe_tokenize(w, merges, alphabet)}')

hug    → ['hug']
hugs   → ['hug', 's']
pug    → ['p', 'ug']
bun    → ['b', 'un']
bug    → ['b', 'ug']
pugs   → ['p', 'ug', 's']
unhug  → ['un', 'hug']
mug    → ['[UNK]', 'ug']
thug   → ['[UNK]', 'hug']


- `bug` · `pugs` · `unhug` 은 학습 데이터에 없던 단어지만 조각(`b + ug`, `p + ug + s`, `un + hug`)으로 표현된다 → **OOV 해결**
- `mug` · `thug` 의 `m` · `t` 는 초기 문자 사전에 없어서 `[UNK]` — 그래서 실제 토크나이저는 말뭉치의 문자들로 초기 사전을 넉넉히 만들고 시작한다(교재 p.30 ②). 단, `BertWordPieceTokenizer` 는 기본값으로 **많이 나온 1,000자만** 남겨서(`limit_alphabet`) 한글처럼 글자 종류가 많으면 드문 글자는 여전히 `[UNK]` 가 된다 ([`wordpiece_vocab_nsmc.ipynb`](./wordpiece_vocab_nsmc.ipynb) 10절). GPT-2 의 byte-level BPE 는 글자 대신 **바이트 256개**로 시작해서 `[UNK]` 가 아예 없다

## 5. 라이브러리로 확인 — Hugging Face `tokenizers` BPE

In [6]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

corpus = [w for w, f in word_freq.items() for _ in range(f)]     # 단어를 빈도만큼 반복한 말뭉치
tok = Tokenizer(models.BPE(unk_token='[UNK]'))
tok.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.BpeTrainer(vocab_size=11, special_tokens=[], show_progress=False)
tok.train_from_iterator(corpus, trainer)

print('라이브러리 단어 사전 :', sorted(tok.get_vocab(), key=tok.get_vocab().get))
print('직접 구현 단어 사전 :', vocab_bpe)
for w in ['hugs', 'pug', 'bun', 'unhug']:
    print(f'{w:6s} → {tok.encode(w).tokens}')

라이브러리 단어 사전 : ['b', 'g', 'h', 'n', 'p', 's', 'u', 'ug', 'un', 'hug', 'pun']
직접 구현 단어 사전 : ['b', 'g', 'h', 'n', 'p', 's', 'u', 'ug', 'un', 'hug', 'pun']
hugs   → ['hug', 's']
pug    → ['p', 'ug']
bun    → ['b', 'un']
unhug  → ['un', 'hug']


## 6. BPE vs WordPiece — 무엇을 합치나 (교재 p.27)

| | BPE | WordPiece |
|---|---|---|
| 병합 기준 | 쌍의 **빈도** $f(a, b)$ 가 최대 | **점수** $\dfrac{f(a, b)}{f(a)\,f(b)}$ 가 최대 |
| 의미 | 자주 붙어 나오는 쌍 | 따로따로는 드문데 **함께 나올 때가 많은** 쌍 |
| 서브워드 표시 | 단어 **앞** 공백을 `Ġ` 로 표시 (GPT-2) | 단어 **중간 · 끝** 조각에 `##` |
| 사용 모델 | GPT-2 · RoBERTa | **BERT** |

In [7]:
def wordpiece_scores(splits, word_freq):
    unit = Counter()
    for w, toks in splits.items():
        for t in toks:
            unit[t] += word_freq[w]                      # 각 토큰(문자)의 빈도
    stats = pair_stats(splits, word_freq)
    return {p: f / (unit[p[0]] * unit[p[1]]) for p, f in stats.items()}, stats


start = {w: tuple(w) for w in word_freq}
scores, stats = wordpiece_scores(start, word_freq)
table = sorted(scores, key=scores.get, reverse=True)
print(f'{"쌍":10s} {"빈도":>4s} {"WordPiece 점수":>14s}')
for p in table:
    print(f'{str(p):10s} {stats[p]:4d} {scores[p]:14.4f}')
print()
print('BPE 첫 병합       :', max(stats, key=stats.get))
print('WordPiece 첫 병합 :', table[0])

쌍            빈도   WordPiece 점수
('g', 's')    5         0.0500
('h', 'u')   15         0.0278
('u', 'g')   20         0.0278
('p', 'u')   17         0.0278
('u', 'n')   16         0.0278
('b', 'u')    4         0.0278

BPE 첫 병합       : ('u', 'g')
WordPiece 첫 병합 : ('g', 's')


같은 데이터인데 첫 병합이 다르다 — BPE 는 가장 흔한 `(u, g)`(20회), WordPiece 는 빈도 5 뿐인 `(g, s)` 를 고른다. `s` 는 `hugs` 에만 있어서 **나올 때마다 항상 `g` 뒤에** 붙어 있으니 '함께 나오는 정도' 점수가 0.05 로 가장 높다. 나머지 쌍은 모두 0.0278 로 같다.

> **📝 고친 부분** (교재 p.27 WordPiece 학습 과정 "가장 빈번하게 등장하는 바이그램 쌍을 반복적으로 병합")
> - 그 설명은 **BPE** 의 기준이다. WordPiece 는 빈도가 아니라 **빈도 ÷ (각 조각 빈도의 곱)** 점수(가능도를 가장 많이 올리는 쌍)로 병합한다. 사전 구성 방식(병합 반복 · 드문 단어는 서브워드로)은 같다